In [1]:
#导包
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
import torch.optim as optim
import jieba 
import time


d:\Anaconda_envs\envs\aitest01\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
#获取词表



unique_words,all_words=[],[]
with open('可爱女人_歌词.txt','r',encoding='utf-8') as f:
    for line in f:
        
        words=jieba.lcut(line) #使用jieba库进行分词，lcut函数将文本分割成一个个单词，并返回一个列表
        all_words.append(words)
        
        for word in words:
            unique_words.append(word) #将分词结果添加到unique_words列表中，unique_words列表用于存储所有的单词，包括重复的单词
    unique_words=list(set(unique_words)) #去重
    print(unique_words)
word_count=len(unique_words)
word_to_index={word:i for i,word in enumerate(unique_words)} #构建词表，字典形式，Key是词，value是索引
print(word_to_index)

#歌词用词表索引表示
idx=[] #各行的索引列表
for words in all_words:
    temp=[]
    for word in words:
        temp.append(word_to_index[word]) #获取每行的索引
    idx.extend(temp) #extend方法是将一个可迭代对象中的元素添加到列表中，而不是将整个对象作为一个元素添加到列表中
print(len(idx))
print(word_to_index['\n'])
print(idx)







Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\icyw\AppData\Local\Temp\jieba.cache
Loading model cost 0.594 seconds.
Prefix dict has been built successfully.


['透明', '想想', '的', '去', '直升机', '宇宙', '可爱', '开始', '而', '小小的', '让', '尽全力', '和', '世界', '全力', '心疼', '小', '蚂蚁', '只是', '女人', '感动', '漂亮', '融化', '在', '疯狂', '这样', '地心引力', '坏坏', '每天', '甜蜜', '我', '命运', '碰到', '有', '但', '保护', '一起', '想要', '温柔', '\n', '只', '里', '面红', '飞到', '大', '要', '你', '银河', '相信', '感谢', '着']
{'透明': 0, '想想': 1, '的': 2, '去': 3, '直升机': 4, '宇宙': 5, '可爱': 6, '开始': 7, '而': 8, '小小的': 9, '让': 10, '尽全力': 11, '和': 12, '世界': 13, '全力': 14, '心疼': 15, '小': 16, '蚂蚁': 17, '只是': 18, '女人': 19, '感动': 20, '漂亮': 21, '融化': 22, '在': 23, '疯狂': 24, '这样': 25, '地心引力': 26, '坏坏': 27, '每天': 28, '甜蜜': 29, '我': 30, '命运': 31, '碰到': 32, '有': 33, '但': 34, '保护': 35, '一起': 36, '想要': 37, '温柔': 38, '\n': 39, '只': 40, '里': 41, '面红': 42, '飞到': 43, '大': 44, '要': 45, '你': 46, '银河': 47, '相信': 48, '感谢': 49, '着': 50}
268
39
[37, 33, 4, 39, 37, 12, 46, 43, 5, 3, 39, 37, 12, 46, 22, 23, 36, 39, 22, 23, 47, 41, 39, 30, 28, 28, 28, 23, 1, 1, 50, 46, 39, 25, 2, 29, 39, 10, 30, 7, 48, 31, 39, 49, 26, 39, 10, 30, 32, 46, 39, 21, 2, 10

In [16]:
#数据预处理，构建数据集
#定义数据集类
class LyricsDataset(Dataset):
    def __init__(self,idx,seq_len):
        self.idx=idx #初始化词索引
        self.seq_len=seq_len #每个样本的序列长度
        self.word_count=len(word_to_index) #词汇表大小
        self.number=self.word_count // self.seq_len #样本数量
#当使用len()方法时，返回样本数量
    def __len__(self):
        return self.number
#当使用obj[index]时，返回第index个样本
    def __getitem__(self,index):
        #index:样本的索引
        start=min(max(index,0),self.word_count-self.seq_len-1) #确保索引在有效范围内
        end=start+self.seq_len
        x= self.idx[start:end] #输入序列
        y= self.idx[start+1:end+1] #标签是输入序列的下一个词
        return torch.tensor(x),torch.tensor(y) #返回输入序列和标签 

#测试
dataset=LyricsDataset(idx,10)
print(dataset[0])


(tensor([30, 26, 45,  9, 30, 39,  1, 10, 40,  3]), tensor([26, 45,  9, 30, 39,  1, 10, 40,  3,  9]))


In [41]:

#搭建神经网络
class TextGenerator(nn.Module):
    def __init__(self,word_count):
        super().__init__()
        self.ebd=nn.Embedding(word_count,128) #词嵌入层，将每个词映射为128维向量
        self.rnn=nn.RNN(128,256,1,batch_first=True) #RNN层，输入维度128，隐藏层维度256，1层，batch_first=True表示输入和输出的第一个维度分别表示批量大小和序列长度
        self.fc=nn.Linear(256,word_count) #全连接层，将隐藏层输出映射为词汇表大小
    def forward(self,x,hidden):
        x=self.ebd(x)
        y,hidden=self.rnn(x,hidden)
        y=self.fc(y) #将RNN输出映射为词汇表大小
        return y,hidden
    def init_hidden(self,batch_size):
        return torch.zeros(1,batch_size,256) #初始化隐藏状态为0,各维度分别表示层数、批量大小、隐藏层维度


        

    

        

In [58]:
#模型训练
model=TextGenerator(word_count)
dataloader=DataLoader(dataset,batch_size=5,shuffle=True) #创建数据加载器，参数：数据集、批量大小、是否随机打乱
optimizer=optim.Adam(model.parameters(),lr=0.001) #创建优化器，参数：模型参数、学习率
criterion=nn.CrossEntropyLoss() #创建损失函数，参数：无
epochs=1000
for epoch in range(epochs):
    model.train()
    total_loss,total_correct,total_samples=0,0,0
    start=time.time()
    for x,y in dataloader:
        hidden=model.init_hidden(x.shape[0]) #初始化隐藏状态
        y_pred,hidden=model(x,hidden)
        optimizer.zero_grad()
        loss=criterion(y_pred.view(-1,word_count),y.view(-1)) #将y_pred和y转换为1维向量，计算损失
        # print(y_pred,y)
        loss.backward()
        optimizer.step()
        total_samples+=x.shape[0]*x.shape[1]
        total_loss+=loss.item()*x.shape[0]*x.shape[1]
        total_correct+=(torch.argmax(y_pred,dim=-1)==y).sum()
    end=time.time()
    #计算准确率
    accuracy=total_correct/total_samples
    #打印训练信息
    print(f"epoch:{epoch+1}/{epochs},loss:{total_loss/total_samples:.4f},accuracy:{accuracy:.4f},time:{time.time()-start:.2f}s")

    


epoch:1/1000,loss:3.9775,accuracy:0.0000,time:0.03s
epoch:2/1000,loss:3.3571,accuracy:0.4800,time:0.02s
epoch:3/1000,loss:2.7725,accuracy:0.9400,time:0.02s
epoch:4/1000,loss:2.2358,accuracy:0.9600,time:0.02s
epoch:5/1000,loss:1.7606,accuracy:0.9400,time:0.03s
epoch:6/1000,loss:1.3591,accuracy:0.9400,time:0.02s
epoch:7/1000,loss:1.0373,accuracy:0.9400,time:0.02s
epoch:8/1000,loss:0.7922,accuracy:0.9400,time:0.02s
epoch:9/1000,loss:0.6132,accuracy:0.9400,time:0.02s
epoch:10/1000,loss:0.4859,accuracy:0.9400,time:0.02s
epoch:11/1000,loss:0.3957,accuracy:0.9400,time:0.02s
epoch:12/1000,loss:0.3311,accuracy:0.9400,time:0.02s
epoch:13/1000,loss:0.2834,accuracy:0.9600,time:0.02s
epoch:14/1000,loss:0.2469,accuracy:0.9600,time:0.02s
epoch:15/1000,loss:0.2180,accuracy:0.9600,time:0.02s
epoch:16/1000,loss:0.1943,accuracy:0.9600,time:0.01s
epoch:17/1000,loss:0.1747,accuracy:0.9600,time:0.01s
epoch:18/1000,loss:0.1586,accuracy:0.9600,time:0.01s
epoch:19/1000,loss:0.1455,accuracy:0.9600,time:0.01s
ep

In [36]:
#保存模型
torch.save(model.state_dict(),'text_generator.pth')

In [60]:
#测试模型
model=TextGenerator(word_count) #创建模型对象
model.load_state_dict(torch.load('text_generator.pth')) #加载模型参数
model.eval()

hidden=model.init_hidden(1) #初始化隐藏状态，批量大小为1，这里的批量大小表示一次生成一个句子
word_idx=word_to_index['女人'] #将'女人'转换为索引
seq=[word_idx] #将'女人'的索引添加到序列中
for i in range(20):
        output,hidden=model(torch.tensor([word_idx]).unsqueeze(0),hidden) #将当前词索引转换为张量，添加批量维度，输入模型
        print(output.shape)
        word_idx=torch.argmax(output).item() #获取输出中概率最大的索引，转换为Python整数
        seq.append(word_idx)#将索引添加到序列中
for idx in seq:
        print(unique_words[idx],end='') #将索引转换为对应的单词，并打印





torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
torch.Size([1, 1, 51])
女人有直升机
想要和你飞到宇宙去
想要和你融化宇宙去
想要和你